# Transformer

## Setup
Instalamos dependencias y dejamos accesible el paquete `src` (donde vive `TransformerTrendModel`, `build_features`, `make_windows`, etc.). Ajustar `REPO_URL` / rama segun corresponda, o comentar este cell si `src` ya esta disponible (por ejemplo corriendo el notebook desde la raiz del repo).

In [1]:
!pip install torch pandas numpy matplotlib seaborn requests

Defaulting to user installation because normal site-packages is not writeable


In [2]:
# Si el notebook no corre desde la raiz del repo, clonamos el repositorio para tener
# el paquete "src" importable (TransformerTrendModel, build_features, make_windows, etc.)
import os
import sys

REPO_URL = "https://github.com/<tu-usuario>/<tu-repo>.git"  # <-- ajustar
REPO_DIR = "/content/repo"

if not os.path.isdir(REPO_DIR):
  os.system(f"git clone {REPO_URL} {REPO_DIR}")

if REPO_DIR not in sys.path:
  sys.path.append(REPO_DIR)

sh: line 1: tu-usuario: No such file or directory


In [ ]:
import json
import math
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
import torch

from lstm import build_features, make_windows
from transformer import TransformerTrainConfig, TransformerTrendModel

ModuleNotFoundError: No module named 'src'

## Preprocessing
Aqui definimos las funciones que utilizaremos, incluyendo el entrenamiento del Transformer

In [ ]:
def get_ticker_data(ticker):
  """
  Given a ticker name returns a pandas dataframe with the historical information
  """
  r = requests.post(f"https://data-colector.onrender.com/historical-data/{ticker}")
  cuerpo = json.loads(r.text)
  return pd.DataFrame(cuerpo["data"])

In [ ]:
# Global dict to keep in memory all tickers fetched
ticker_stats = {}

def ticker_data(ticker, use_cache=True):
  global ticker_stats
  if ticker in ticker_stats and use_cache:
    return ticker_stats[ticker]
  else:
    ticker_stats[ticker] = get_ticker_data(ticker)
    return ticker_stats[ticker]

Entrenamos el Transformer pooleando varios tickers a la vez (asi esta pensado el modelo: `fit` recibe un `dict {ticker: dataframe}`), a diferencia de ARIMA que se entrena un ticker a la vez.

In [ ]:
def train_transformer(histories: dict[str, pd.DataFrame], **config_overrides) -> TransformerTrendModel:
  """
  Given a dict {ticker: dataframe_ohlcv} and config overrides, returns a trained TransformerTrendModel
  """
  config = TransformerTrainConfig(**config_overrides)
  model = TransformerTrendModel(config=config)
  model.fit(histories)
  return model

Para evaluar reutilizamos el mismo feature engineering que usa el modelo internamente (`build_features` + `make_windows`), y llamamos a la red ya entrenada para comparar retorno predicho contra retorno real, ticker por ticker.

In [ ]:
def predict_windows(model: TransformerTrendModel, df: pd.DataFrame):
  """
  Given a trained model and a raw OHLCV dataframe, returns (pred, real) arrays de retornos
  logaritmicos, uno por cada ventana que se puede armar con ese dataframe
  """
  cfg = model.config
  feats, target = build_features(df, cfg.horizon)
  X, y = make_windows(feats, target, cfg.window)
  if len(X) == 0:
    return np.array([]), np.array([])

  x_t = model._to_tensor(model._scale_features(X))
  model._net.eval()
  with torch.no_grad():
    scaled = model._net(x_t).numpy()
  pred = scaled * model.target_std + model.target_mean
  return pred, y

Utilizamos el error RMSE (sobre el retorno logaritmico predicho) para calcular distancias
$$\sqrt[]{\frac{1}{N}\sum_{i = 0}^N{(y_i - y'_i)^2}}$$

In [ ]:
def rmse(y_true, y_pred):
  n = len(y_true)
  if n == 0:
    return float("nan")
  differences = (y_true - y_pred) ** 2
  return math.sqrt((differences / n).sum())

Accuracy

$$Accuracy = \frac{1}{N}\sum_{i = 0}^{N}{ (signo(y_i) == signo(y_i'))}$$
Analiza el porcentaje de veces que el modelo predice correctamente el signo del retorno (sube/baja), analogo a la accuracy usada para ARIMA pero directamente sobre el retorno en vez de tener que reconstruir el precio.

Igual que en ARIMA, tambien separamos en "Up Accuracy" y "Down Accuracy": el porcentaje de aciertos cuando el valor real sube y cuando baja, respectivamente.

In [ ]:
def accuracy(y_true, y_pred):
  """
  Porcentaje de veces que el signo del retorno predicho coincide con el signo del retorno real
  """
  if len(y_true) == 0:
    return float("nan")
  return float(np.mean(np.sign(y_true) == np.sign(y_pred)))

In [ ]:
def up_accuracy(y_true, y_pred):
  """
  Accuracy restringida a los casos donde el valor real subio
  """
  up = y_true > 0
  if up.sum() == 0:
    return 1.0
  return float(np.mean(y_pred[up] > 0))

In [ ]:
def down_accuracy(y_true, y_pred):
  """
  Accuracy restringida a los casos donde el valor real bajo
  """
  down = y_true <= 0
  if down.sum() == 0:
    return 1.0
  return float(np.mean(y_pred[down] <= 0))

### Implementacion de tests

A diferencia de ARIMA (que se entrena y testea ticker por ticker), el Transformer se entrena una unica vez pooleando todos los tickers, dejando afuera del entrenamiento los ultimos `n_to_predict` dias de cada uno. Despues evaluamos por ticker sobre ese tramo final, que el modelo nunca vio durante el entrenamiento.

In [ ]:
def get_test_results(tickers: list[str], n_to_predict: int, use_cache: bool = True, **config_overrides):
  """
  Entrena el Transformer pooleando "tickers" (dejando afuera los ultimos n_to_predict dias de
  cada uno) y devuelve, por ticker, una tupla (y_real, y_pred) sobre esos dias dejados afuera
  """
  train_histories = {}
  full_histories = {}
  for ticker in tickers:
    data = ticker_data(ticker, use_cache=use_cache)
    full_histories[ticker] = data
    train_histories[ticker] = data.iloc[: len(data) - n_to_predict].reset_index(drop=True)

  model = train_transformer(train_histories, **config_overrides)

  test_results = []
  for ticker in tickers:
    pred, real = predict_windows(model, full_histories[ticker])
    # Nos quedamos solo con las ultimas n_to_predict ventanas: las que caen en el tramo
    # que el modelo no vio durante el entrenamiento
    test_results.append((real[-n_to_predict:], pred[-n_to_predict:]))

  return model, test_results

In [ ]:
def calculate_accuracy(test_results, acc_type="general"):
  match acc_type:
    case "up":
      accuracy_result = [up_accuracy(*ticker) for ticker in test_results]
    case "down":
      accuracy_result = [down_accuracy(*ticker) for ticker in test_results]
    case "general":
      accuracy_result = [accuracy(*ticker) for ticker in test_results]
  return accuracy_result

In [ ]:
def calculate_rmse(test_results):
  return [rmse(*ticker) for ticker in test_results]

# Pruebas

## RMSE

### RMSE en relacion al horizonte de prediccion

In [ ]:
# Utilizamos los mismos tickers del panel general que en ARIMA, para poder comparar
tickers = ["BYMA", "YPFD", "SUPV", "GGAL", "BBAR"]
# Utilizamos distintos horizontes de prediccion
ahead_time = [2, 5, 10, 15, 30]

# epochs bajo para que el sweep no tarde una eternidad; para un entrenamiento final conviene subirlo
results = []
for ahead in ahead_time:
  _, test_results = get_test_results(tickers, n_to_predict=ahead, horizon=ahead, epochs=15)
  results.append(calculate_rmse(test_results))

# Transponemos para tener la misma forma que en ARIMA: filas=tickers, columnas=horizonte
results = np.array(results).T.tolist()

In [ ]:
fig, ax = plt.subplots()
im = ax.imshow(results)

ax.set_xticks(range(len(ahead_time)), labels=ahead_time, rotation=45)
ax.set_yticks(range(len(tickers)), labels=tickers)

for i in range(len(tickers)):
    for j in range(len(ahead_time)):
        text = ax.text(j, i, round(results[i][j], 4),
                       ha="center", va="center", color="black")

ax.set_title("RMSE (retorno log) segun horizonte de prediccion contra tickers")
fig.tight_layout()
plt.show()

### RMSE segun tamaño de ventana (window) y tiempo de entrenamiento

In [ ]:
tickers = ["BYMA", "YPFD", "SUPV", "GGAL", "BBAR"]

ahead_time = 5
windows = [15, 30, 45, 60]

results = []
durations = []
for w in windows:
  start = time.process_time()
  print(f"Starting test window: {w}")
  _, test_results = get_test_results(tickers, n_to_predict=ahead_time, window=w, horizon=ahead_time, epochs=15)
  results.append(calculate_rmse(test_results))
  durations.append(time.process_time() - start)

In [ ]:
results_t = np.array(results).T.tolist()

fig, ax = plt.subplots()
im = ax.imshow(results_t)

ax.set_xticks(range(len(windows)), labels=windows, rotation=45)
ax.set_yticks(range(len(tickers)), labels=tickers)

for i in range(len(tickers)):
    for j in range(len(windows)):
        text = ax.text(j, i, round(results_t[i][j], 4),
                       ha="center", va="center", color="black")

ax.set_title("RMSE (retorno log) segun tamaño de ventana contra tickers")
fig.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots()
ax.plot(windows, durations, marker="o")
ax.set_xlabel("Window")
ax.set_ylabel("Segundos")
ax.set_title("Tiempo de entrenamiento (pooled) segun tamaño de ventana")
plt.show()

### Analisis de RMSE sobre todas las acciones del panel general

In [ ]:
PANEL_LIDER = ["YPFD", "GGAL", "BMA", "TECO2", "PAMP", "BBAR", "TGSU2",
               "TXAR", "CEPU", "ALUA", "EDN", "BYMA", "LOMA", "SUPV",
               "CVH", "TRAN", "CRES", "HARG", "COME", "MIRG", "TGNO4",
               "VALO", "AGRO"]

model, results = get_test_results(PANEL_LIDER, n_to_predict=5, horizon=5, epochs=40)

In [ ]:
fig, axs = plt.subplots(1, 1, sharey=True, tight_layout=True)
axs.hist(calculate_rmse(results), bins=5)

axs.set_title("RMSE")

plt.show()

## Accuracy

### Accuracy segun horizonte de prediccion

In [ ]:
tickers = ["BYMA", "YPFD", "SUPV", "GGAL", "BBAR"]
ahead_time = [2, 5, 10, 15, 30]

times = []
for ahead in ahead_time:
  _, test_results = get_test_results(tickers, n_to_predict=ahead, horizon=ahead, epochs=15)
  times.append(calculate_accuracy(test_results))

times = np.array(times).T.tolist()

In [ ]:
fig, ax = plt.subplots()
im = ax.imshow(times)

ax.set_xticks(range(len(ahead_time)), labels=ahead_time, rotation=45)
ax.set_yticks(range(len(tickers)), labels=tickers)

for i in range(len(tickers)):
    for j in range(len(ahead_time)):
        text = ax.text(j, i, round(times[i][j], 3),
                       ha="center", va="center", color="black")

ax.set_title("Accuracy segun horizonte de prediccion contra tickers")
fig.tight_layout()
plt.show()

### Accuracy segun tamaño de ventana

In [ ]:
tickers = ["BYMA", "YPFD", "SUPV", "GGAL", "BBAR"]
ahead_time = 5
windows = [15, 30, 45, 60]

times = []
for w in windows:
  _, test_results = get_test_results(tickers, n_to_predict=ahead_time, window=w, horizon=ahead_time, epochs=15)
  times.append(calculate_accuracy(test_results))

In [ ]:
fig, ax = plt.subplots()
im = ax.imshow(times)

ax.set_xticks(range(len(tickers)), labels=tickers, rotation=45)
ax.set_yticks(range(len(windows)), labels=windows)

for i in range(len(windows)):
    for j in range(len(tickers)):
        text = ax.text(j, i, round(times[i][j], 3),
                       ha="center", va="center", color="black")

ax.set_title("Accuracy segun tamaño de ventana contra tickers")
fig.tight_layout()
plt.show()

### Analisis general de Accuracy sobre todos los tickers del panel general

In [ ]:
PANEL_LIDER = ["YPFD", "GGAL", "BMA", "TECO2", "PAMP", "BBAR", "TGSU2",
               "TXAR", "CEPU", "ALUA", "EDN", "BYMA", "LOMA", "SUPV",
               "CVH", "TRAN", "CRES", "HARG", "COME", "MIRG", "TGNO4",
               "VALO", "AGRO"]

model, results = get_test_results(PANEL_LIDER, n_to_predict=5, horizon=5, epochs=40)

In [ ]:
fig, axs = plt.subplots(1, 3, sharey=True, tight_layout=True)
axs[0].hist(calculate_accuracy(results), bins=10, range=(0, 1))
axs[1].hist(calculate_accuracy(results, acc_type="up"), bins=10, range=(0, 1))
axs[2].hist(calculate_accuracy(results, acc_type="down"), bins=10, range=(0, 1))

axs[0].set_title("Accuracy")
axs[1].set_title("Up Accuracy")
axs[2].set_title("Down Accuracy")

plt.show()

In [ ]:
print(f"Promedio de accuracy       {sum(calculate_accuracy(results, acc_type='general')) / len(results)}")
print(f"Promedio de up accuracy    {sum(calculate_accuracy(results, acc_type='up')) / len(results)}")
print(f"Promedio de down accuracy  {sum(calculate_accuracy(results, acc_type='down')) / len(results)}")

### Metricas propias del modelo (val set interno)
`model.metrics` ya trae `val_mae_logret` y `val_directional_accuracy` calculados sobre el val set interno que separa `fit` (distinto al holdout de `n_to_predict` dias que armamos arriba). Sirve como chequeo cruzado rapido.

In [ ]:
model.metrics